# IMDB Reviews Dataset

## Overview

The IMDB Reviews dataset is a collection of 50,000 movie reviews from the Internet Movie Database (IMDB),
with an equal split between training and test sets.

## Dataset Characteristics

- **Total Reviews**: 50,000
- **Training Set**: 25,000 reviews
- **Test Set**: 25,000 reviews
- **Labels**: Binary classification (0 = Negative, 1 = Positive)
- **Vocabulary Size**: 10,000 most frequent words

## Data Format

- Reviews are encoded as sequences of integers
- Each integer represents a specific word (word index)
- Common indices:
  - 1 = Start of review
  - 2 = Unknown/OOV (Out of Vocabulary) words
  - 3+ = Actual word indices

## Usage in Deep Learning

This dataset is commonly used for:

- Binary sentiment classification tasks
- Natural Language Processing (NLP) demonstrations
- Training and evaluating RNN, LSTM, and other sequence models
- Benchmarking text analysis models

## Current Status

- X_train shape: (25000,) - Training reviews
- y_train shape: (25000,) - Training labels
- X_test shape: (25000,) - Test reviews
- y_test shape: (25000,) - Test labels


# Training


In [47]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
VOCAB_SIZE = 10000  # Number of words to consider as features

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f"Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}")

print(f"Test data shape: {X_test.shape}, Test labels shape: {y_test.shape}")

Training data shape: (25000,), Training labels shape: (25000,)
Test data shape: (25000,), Test labels shape: (25000,)


In [25]:
import random

reverse_word_index = {value: key for (key, value) in imdb.get_word_index().items()}

sample_review = X_train[random.randint(0, len(X_train) - 1)]  # any random value
" ".join(reverse_word_index.get(i - 3, "?") for i in sample_review)

"? this film was more effective in ? me of a ? conspiracy than a muslim one and i'm jewish br br ? go to ? school read an ? ? year ? these ? ? presented in a ? way might prove ? but by offering no ? of possible ? arguments nor ? or any at all ? few sources and each of dubious origin makes the argument an ? ? br br and thank goodness for that i wouldn't want anyone to leave the theatre believing any of this racist ? br br a good lesson for me and hopefully a ? tale for you to actually read about a film before seeing it"

In [27]:
MAX_LEN = 500  # Maximum length of review

X_train = sequence.pad_sequences(X_train, maxlen=MAX_LEN)
X_test = sequence.pad_sequences(X_test, maxlen=MAX_LEN)

In [32]:
X_train

array([[   0,    0,    0, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]],
      shape=(25000, 500), dtype=int32)

In [33]:
X_test

array([[   0,    0,    0, ...,   14,    6,  717],
       [   0,    0,    0, ...,  125,    4, 3077],
       [  33,    6,   58, ...,    9,   57,  975],
       ...,
       [   0,    0,    0, ...,   21,  846, 5518],
       [   0,    0,    0, ..., 2302,    7,  470],
       [   0,    0,    0, ...,   34, 2005, 2643]],
      shape=(25000, 500), dtype=int32)

## Training SimpleRNN


In [48]:
EMBEDDING_DIM = 128  # Dimension of the embedding vector

model = Sequential()

model.add(Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN))
model.add(SimpleRNN(128, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=10,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
)

Epoch 1/10


d:\Workspace\ai-ml-dl\ml-ds-krish-naik\23-rnn-project\venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


782/782 ━━━━━━━━━━━━━━━━━━━━ 113s 143ms/step - accuracy: 0.6146 - loss: 786.4805 - val_accuracy: 0.5528 - val_loss: 0.9607
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 118s 150ms/step - accuracy: 0.6414 - loss: 184.6030 - val_accuracy: 0.6587 - val_loss: 0.6000
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 108s 138ms/step - accuracy: 0.7601 - loss: 0.4929 - val_accuracy: 0.7499 - val_loss: 0.5147
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 107s 136ms/step - accuracy: 0.8440 - loss: 0.3576 - val_accuracy: 0.7831 - val_loss: 0.4633
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 99s 127ms/step - accuracy: 0.8932 - loss: 0.2649 - val_accuracy: 0.8102 - val_loss: 0.4611
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 104s 133ms/step - accuracy: 0.9128 - loss: 0.2262 - val_accuracy: 0.8032 - val_loss: 0.4676
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 107s 137ms/step - accuracy: 0.9330 - loss: 0.1819 - val_accuracy: 0.8007 - val_loss: 0.5378
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 121s 154ms/step - accuracy: 0.9485 - loss: 0

In [50]:
model.save("./models/imdb_rnn_model.h5")

# Prediction


In [51]:
model = tf.keras.models.load_model("./models/imdb_rnn_model.h5")

In [75]:
def preprocess_review(review):
    tokens = review.lower().split()
    word_index = imdb.get_word_index()

    indexed_review = [
        word_index.get(word, 2) + 3 if word in word_index else 2 for word in tokens
    ]

    padded_review = sequence.pad_sequences([indexed_review], maxlen=MAX_LEN)
    return padded_review

In [76]:
def predict_sentiment(review):
    processed_review = preprocess_review(review)
    prediction = model.predict(processed_review)
    sentiment = "Positive" if prediction[0][0] > 0.5 else "Negative"
    return sentiment, prediction[0][0]

In [77]:
example_review = "I love this movie. this movie was great"
sentiment, confidence = predict_sentiment(example_review)
print(f"Review: {example_review}")
print(f"Predicted Sentiment: {sentiment} (Confidence: {confidence:.4f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Review: I love this movie. this movie was great
Predicted Sentiment: Negative (Confidence: 0.0039)
